In [4]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 100
lr = 1e-3
ss = 11
device = "cpu"

class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
        return [optimizer], [scheduler]

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, hidden_sizes=architecture, 
                   learning_rate=lr, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name="final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 50:  26%|██▌       | 823/3178 [00:21<01:00, 39.04it/s, v_num=9, val_acc=0.963] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\Braxt\miniconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 25
lr = 1e-3
ss = 11
device = "cpu"

class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
        return [optimizer], [scheduler]

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, hidden_sizes=architecture, 
                   learning_rate=lr, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name="final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------
0 | network   | Sequential         | 706 K  | train | 0    
1 | loss_func | CrossEntropyLoss   | 0      | train | 0    
2 | val_acc   | MulticlassAccuracy | 0      | train | 0    
3 | test_acc  | MulticlassAccuracy | 0      | train | 0    
-----------------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00, 10.96it/s]

c:\Users\braxt\miniconda3\envs\CSCI4120\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\braxt\miniconda3\envs\CSCI4120\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 24: 100%|██████████| 3178/3178 [00:55<00:00, 57.17it/s, v_num=12, val_acc=0.962]

`Trainer.fit` stopped: `max_epochs=25` reached.


Epoch 24: 100%|██████████| 3178/3178 [00:55<00:00, 57.14it/s, v_num=12, val_acc=0.962]


Restoring states from the checkpoint path at my_logs\final\version_12\checkpoints\epoch=24-step=79450.ckpt
Loaded model weights from the checkpoint at my_logs\final\version_12\checkpoints\epoch=24-step=79450.ckpt
c:\Users\braxt\miniconda3\envs\CSCI4120\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:04<00:00, 217.05it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc             0.960723876953125
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 25, 'learning_rate': 0.001, 'scheduler': 'StepLR', 'step': 11, 'training_time': 1249.2193984985352, 'test_accuracy': 0.960723876953125}
